In [ ]:
import numpy as np
import os, sys
import matplotlib.pyplot as plt
import h5py
import imageio as imio

from PyGEECSPlotter.scan_data_analyzers.scan_data_analysis import ScanDataAnalyzer
from PyGEECSPlotter.analyzers.image_analysis import ImageAnalyzer
from PyGEECSPlotter.utils.widgets import get_selection_box


In [ ]:
# Setting the diagnostic and analysis, display options for running 
# the analysis and for displaying data


diagnostic='HTT_Phasics_FileCopy'
file_ext='.h5'


analyzer_dict = {
    # 1) Saturation
    'return_n_saturated': False,  # Whether to count saturated pixels
    'bit_depth': 12,  # Bit depth of the image
    'saturation_threshold': None,  # Custom saturation threshold

    # 2) Background subtraction
    'bg_file': True,  # Whether to use a background file
    'bg_const': 0,  # Constant background subtraction value
}

display_dict = {'cbar_label' : 'Phase (n. waves)',
                'cmap' : 'RdBu'
               }

# Define the Analyzer




In [ ]:
class Phasicsh5Analyzer(ImageAnalyzer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def load_data(self, filename):
        with h5py.File(filename, "r") as f:
            return f["phase"][:]

    def analyze_data(self, data, analyzer_dict=None, bg=None):
        if analyzer_dict is None:
            analyzer_dict = self.analyzer_dict

        if bg is not None:
            data = data - bg

        data = data-np.nanmean(data)

        return_dict = Phasicsh5Analyzer.compute_phase_shifts( data )

        return data, return_dict

    def write_analyzed_data(self, bin_filepath, data, nan_value=0):
        """
        Write a 2D NumPy array to a binary PNG image and create a scaling information text file.
        
        Parameters:
        - bin_filepath (str): The base file path for the PNG image and scaling text file.
        - data (numpy.ndarray): The input 2D array to be saved.
        - nan_value (float, optional): The scalar value to replace NaN values with. Default is 0.
        
        Returns:
        - None
        
        The function scales the input data to fit within a 16-bit range (0-65535) and saves it as a binary PNG image. 
        The scaling factors (min and max) used for scaling are saved in a text file with the same name.
        """
        
        # Replace NaN values with the specified scalar
        data = np.nan_to_num(data, nan=nan_value)
    
        # Calculate the scaling factors
        data_int = (65535 * ((data - np.min(data)) / np.ptp(data))).astype(np.uint16)
    
        # Create scaling information lines
        lines = ['[Scaling]', 'min = %f' % np.min(data), 'max = %f' % np.max(data)]
    
        # Write scaling information to a text file
        with open(bin_filepath + '.txt', 'w') as f:
            f.write('\n'.join(lines))
    
        # Save the scaled data as a binary PNG image
        imio.imwrite(bin_filepath + '.png', data_int)
        
    # -------------------------------------------------------------------
    # Utility methods (can be overridden or extended in subclasses)
    # -------------------------------------------------------------------

    @staticmethod
    def compute_phase_shifts(data, shifts_when=''):
        max_data = np.nanmax(data)
        min_data = np.nanmin(data)
        ptp = np.nanmax(data) - np.nanmin(data)
        rms = np.nanstd(data)

        return {f'max shift {shifts_when}': max_data,
                f'min shift {shifts_when}': min_data,
                f'ptp shift {shifts_when}': ptp,
                f'rms shift {shifts_when}': rms,
               }

In [ ]:
analyzer = Phasicsh5Analyzer(diagnostic=diagnostic, 
                 file_ext=file_ext, 
                 analyzer_dict=analyzer_dict, 
                 display_dict=display_dict,
                 output_diagnostic=f'{diagnostic}_online',
                 output_file_ext=''
                        )

In [ ]:
# Can use search_replace to change column names to more useful names
# Can use column_math to quickly turn column values into more useful quantites (e.g. stage positions)

# column_math_filename = '/Users/alexpicksley/Documents/GitHub/bella_analysis/user_data/2024-02-29_columnmath_testguiding.txt'
# search_replace_filename = '/Users/alexpicksley/Documents/GitHub/bella_analysis/user_data/2024-02-29_searchreplace.txt'



In [ ]:
# Define which scan to look at 

experiment_dir = r'Z:data/Thomson/'
# experiment_dir = '/Users/alexpicksley/Documents/LBNL/General_PW/GEECS_Testing'
_, _, _, _, _, sfile_disp = get_selection_box(experiment_dir, display_widgets=True)


In [ ]:
# Select sfile from GUI
sfilename = sfile_disp.value

# OR Manually define s-file
sfilename = r'Z:data/Thomson/Y2025\10-Oct\25_1016\analysis\s47.txt'

print("sfilename = r'%s'" %sfilename)

In [ ]:
# Load the Scalar Data for that scan in, and filter any data if desired

scan_data = ScanDataAnalyzer(sfilename=sfilename)

scan_data.load_scan_data(
    # search_replace_filename=search_replace_filename, 
    # column_math_filename=column_math_filename,
    analyzer=analyzer
)

# scan_data.filter_scan_data('HPD stage position', 200, 230, filter_exclusive=False)



In [ ]:
# Automatically load the bg file for that diagnostic

# bg_filename = scan_data.get_bg_file_path(diagnostic, which_scan='last')
# bg = analyzer.load_data(bg_filename)

bg_scan = 55 #scan_data.scan-1
bg_day = scan_data.day-1
bg_sfilename = r'Z:data/Thomson/Y2025\10-Oct\25_1016\analysis\s49.txt'

bg_scan_data = ScanDataAnalyzer( sfilename = bg_sfilename
                            )

bg_scan_data.load_scan_data(
    # search_replace_filename=search_replace_filename, 
    # column_math_filename=column_math_filename,
    analyzer=analyzer
)

bin_numbers, mean_data_per_bin, std_data_per_bin = compute_mean_std_per_bin(bg_scan_data, analyzer)

bg=mean_data_per_bin[0]
plt.imshow(bg)

In [ ]:
# Check one image

idx = np.where(scan_data.data['Shotnumber'] == 8)[0][0]
filename = scan_data.data['%s file_list' %diagnostic][idx]
print(filename)

data = analyzer.load_data(filename)
data, return_dict = analyzer.analyze_data(data, bg=bg)

fig, ax = analyzer.display_data(data, display_dict)

# fig, ax = analyzer.display_lineout(return_dict['x'], return_dict['x_lo'], title=os.path.basename(filename))


In [ ]:
# Analyze the whole scan and save the analyzed scalar values back to the Scalar Data File

add_columns_df = scan_data.analyze_scan(analyzer, 
        bg=bg, 
        display_data=False,
        write_columns_to_sfile=False, 
        overwrite_columns=True, 
        write_analyzed=True,
        add_data=True,
        analysis_label='',
        )

In [ ]:
bin_numbers, mean_data_per_bin, std_data_per_bin = compute_mean_std_per_bin(scan_data, analyzer, bg=bg)


In [ ]:
norm = lambda x: (x - x.min())/(x.max()-x.min())

for i, bin_number in zip( range(len(bin_numbers)), bin_numbers ) :
    fig, ax = analyzer.display_data(mean_data_per_bin[i])
    ny, nx = mean_data_per_bin[i].shape
    scaled_sum_x = norm(np.nansum(mean_data_per_bin[i], axis = 1))*25
    ax.plot(scaled_sum_x, np.arange(ny), color="grey")
    fig.savefig( os.path.join( scan_data.analysis_dir, f'{analyzer.output_diagnostic}_mean_per_bin_Bin{bin_number:02d}.png' ) , dpi=200)
    
    fig, ax = analyzer.display_data(std_data_per_bin[i])
    fig.savefig( os.path.join( scan_data.analysis_dir, f'{analyzer.output_diagnostic}_std_per_bin_Bin{bin_number:02d}.png' ) , dpi=200)
    

In [ ]:
import PyGEECSPlotter.utils.plotting as pygplt
cmap = 'Reds_r'
colors =  pygplt.cmap_to_rgb_array(cmap_name=cmap, num_samples=len(bin_numbers))


In [ ]:
import matplotlib as mpl

norm = mpl.colors.Normalize(vmin=0, vmax=2.5)

fig, ax = plt.subplots(figsize=(4,4), constrained_layout=True)


for i, bin_number in zip( range(len(bin_numbers)), bin_numbers ) :
    ax.plot(np.sum(mean_data_per_bin[i], axis=1), c=colors[i])

ax.set_ylabel("Phase [n. waves]")
ax.set_xlabel("Pixel")
ax.set_title("Longitudinal sum")
plt.title("Transverse probe scan47 (BG: scan49)")
# plt.legend()

sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])

# Add a new axis for the colorbar (x0, y0, width, height)
cax = fig.add_axes([0.67, 0.89, 0.3, 0.03])  # centered at bottom
cbar = fig.colorbar(sm, cax=cax, orientation='horizontal')
cbar.set_label("Delay (ns)")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots()

# Define a colormap and normalization range
cmap = plt.cm.jet
norm = mpl.colors.Normalize(vmin=0, vmax=1)

# Add arbitrary content (no image)
ax.plot([0, 1], [0, 1], 'k--')
ax.set_title("Colorbar added manually")

# Create a dummy ScalarMappable to feed to the colorbar
sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])

# Add a new axis for the colorbar (x0, y0, width, height)
cax = fig.add_axes([0.15, 0.2, 0.02, 0.6])  # left, bottom, width, height in figure coords
cbar = fig.colorbar(sm, cax=cax)
cbar.set_label('Arbitrary Units')


In [ ]:
fig, ax = plt.subplots(1,2, figsize=(10,4))
for i, bin_number in zip( range(len(bin_numbers)), bin_numbers ) :
    ax[1].plot(np.sum(mean_data_per_bin[i], axis=1), label=bin_numbers[i])

ax[1].set_ylabel("Phase [n. waves]")
ax[1].set_xlabel("Pixel")
ax[1].set_title("Longitudinal sum")
plt.suptitle("Transverse probe scan47 (BG: scan49)")
# plt.legend()

In [ ]:
fig, ax = plt.subplots(1,2,

In [ ]:
import numpy as np

def compute_mean_std_per_bin(scan_data, analyzer, bg=None, bin_key='temp Bin number'):
    """
    Compute mean and standard deviation of analyzed data grouped by unique bin numbers.

    Parameters
    ----------
    scan_data : object
        Scan data object with `.data` attribute (DataFrame-like) and `.filter_scan_data()` method.
    analyzer : object
        Analyzer instance with `.load_data()` and `.analyze_data()` methods.
    bg : any, optional
        Background data to pass to `analyzer.analyze_data()`.
    bin_key : str, optional
        Column name to group by (default is 'temp Bin number').

    Returns
    -------
    bin_numbers : np.ndarray
        Unique bin identifiers.
    mean_data_per_bin : np.ndarray
        Mean analyzed data per bin.
    std_data_per_bin : np.ndarray
        Standard deviation of analyzed data per bin.
    """

    bin_numbers = np.unique(scan_data.data[bin_key])
    mean_data_per_bin = []
    std_data_per_bin = []

    for bin_number in bin_numbers:
        tmp_data = scan_data.filter_scan_data(bin_key, bin_number - 0.1, bin_number + 0.1)

        all_data = []
        for i in range(len(tmp_data)):
            filename = tmp_data[f'{analyzer.diagnostic} file_list'][i]
            data = analyzer.load_data(filename)
            data, _ = analyzer.analyze_data(data, bg=bg)
            all_data.append(data)

        all_data = np.asarray(all_data)
        mean_data = np.nanmean(all_data, axis=0)
        std_data = np.nanstd(all_data, axis=0)

        mean_data_per_bin.append(mean_data)
        std_data_per_bin.append(std_data)

    mean_data_per_bin = np.asarray(mean_data_per_bin)
    std_data_per_bin = np.asarray(std_data_per_bin)

    return bin_numbers, mean_data_per_bin, std_data_per_bin
